# Embedding

Sign in as the admin test user, download accepted PDFs from the private bucket, extract each one once per chunk tier, then embed each result. Chunk tiers come from `config/chunkers.py`: fast is a 512-token window, medium is the section chunker at that limit, and slow prepends one situation sentence from the LLM before embedding. Retrieval tiers are not searched here. The parser stays the default, medium Docling.
Set `CHUNK_TIERS_SELECTED` to the subset you want. Each chunk tier is embedded in its own collection, so a later tier does not replace an earlier one. Set `SHOW_TEXT = True` to print one chunk. The last cell deletes those collections and the temporary files.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

load_dotenv(ROOT / ".env")

from helpers import download_supabase_pdfs, new_session_id, preview, timed

from config.chunkers import CHUNK_TIERS, chunk_method_for

SHOW_TEXT = False
CHUNK_TIERS_SELECTED = list(CHUNK_TIERS)
workspace = tempfile.TemporaryDirectory(prefix="nb-embed-")
base_session = new_session_id()
sessions = {tier: f"{base_session}-{tier}" for tier in CHUNK_TIERS_SELECTED}

with timed("download") as setup:
    bucket, accepted, rejected = download_supabase_pdfs(Path(workspace.name))

print(bucket)
print(f"download: {setup['seconds']:.3f}s")
print(f"accepted: {len(accepted)}  rejected: {len(rejected)}")
for name, _size, reason in rejected:
    print(f"{name}\trejected: {reason}")
for tier in CHUNK_TIERS_SELECTED:
    print(f"chunk\t{tier}\t{chunk_method_for(tier)}\t{sessions[tier]}")

In [ ]:
from pipeline.extractor import extract

documents = []
failures = []
for chunk_tier in CHUNK_TIERS_SELECTED:
    chunk_method = chunk_method_for(chunk_tier)
    for path in accepted:
        try:
            with timed("extract") as timing:
                document = await extract(path, chunk_tier=chunk_tier)
            if document.metadata.total_chunks == 0:
                failures.append(
                    (
                        chunk_tier,
                        path.name,
                        "RuntimeError",
                        "produced no text",
                    )
                )
                continue
            documents.append((chunk_tier, chunk_method, document, timing["seconds"]))
        except Exception as exc:
            failures.append((chunk_tier, path.name, type(exc).__name__, str(exc)))

In [ ]:
import statistics

header = (
    "chunk\tchunk_method\tfilename\tpages\tchunks\theadings\tmin\tmedian\tmax\tseconds"
)
print(header)
for chunk_tier, chunk_method, document, seconds in documents:
    lengths = [len(chunk.text) for chunk in document.chunks]
    headings = {tuple(chunk.headings) for chunk in document.chunks if chunk.headings}
    print(
        chunk_tier,
        chunk_method,
        document.metadata.filename,
        document.metadata.total_pages,
        document.metadata.total_chunks,
        len(headings),
        min(lengths),
        statistics.median(lengths),
        max(lengths),
        f"{seconds:.3f}",
        sep="\t",
    )

for chunk_tier, name, exc_type, message in failures:
    print(f"{chunk_tier}\t{name}\tfailure\t{exc_type}\t{message}")

In [ ]:
if SHOW_TEXT and documents:
    chunk_tier, _method, document, _seconds = documents[0]
    chunk = document.chunks[0]
    print(chunk_tier, chunk.page, chunk.headings)
    print(preview(chunk.text, SHOW_TEXT))

In [ ]:
from pipeline.embedder import embed_document

embedded = []
for chunk_tier, chunk_method, document, _extract_seconds in documents:
    with timed("embed") as timing:
        await embed_document(document, sessions[chunk_tier], chunk_tier=chunk_tier)
    embedded.append((chunk_tier, chunk_method, document, timing["seconds"]))

In [ ]:
from pipeline.vector_store import list_sources

print("chunk\tchunk_method\tfilename\tchunks\tsources\tseconds")
for chunk_tier, chunk_method, document, seconds in embedded:
    sources = list_sources(sessions[chunk_tier])
    print(
        chunk_tier,
        chunk_method,
        document.metadata.filename,
        document.metadata.total_chunks,
        ", ".join(sources),
        f"{seconds:.3f}",
        sep="\t",
    )

In [ ]:
from pipeline.embedder import delete_collection

for tier, session_id in sessions.items():
    delete_collection(session_id)
    print(tier, session_id, "deleted")
workspace.cleanup()